# Customer Churn Analysis — Análise de Cancelamento de Clientes

## Objetivo

Analisar dados de clientes de uma empresa de telecomunicações para identificar características e padrões associados ao cancelamento (churn), utilizando Python e Pandas.

## Etapas do Projeto

1. Importação e compreensão da base de dados
2. Exploração e avaliação da qualidade dos dados
3. Tratamento e preparação dos dados
4. Análise da taxa de churn
5. Análise dos fatores associados ao cancelamento
6. Criação de visualizações
7. Identificação dos principais achados
8. Elaboração de recomendações de negócio

In [207]:
import pandas as pd

In [208]:
tabela = pd.read_csv("../data/Telco-Customer-Churn.csv")

In [209]:
tabela.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Exploração e Qualidade dos Dados

In [210]:
tabela.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 2. Tratamento e Preparação dos Dados

In [211]:
pd.to_numeric(tabela["TotalCharges"], errors="coerce").isna().sum()

np.int64(11)

In [212]:
tabela.loc[
    pd.to_numeric(tabela["TotalCharges"], errors="coerce").isna(),
    "TotalCharges"
].unique()

<StringArray>
[' ']
Length: 1, dtype: str

In [213]:
tabela.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [214]:
tabela.duplicated().sum()

np.int64(0)

In [215]:
tabela[tabela.duplicated(keep=False)]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [216]:
tabela[tabela["tenure"] == 0].shape

(11, 21)

In [217]:
tabela["TotalCharges"] = tabela["TotalCharges"].fillna(0)

In [218]:
tabela["TotalCharges"].isna().sum()

np.int64(0)

## 3. Análise da Taxa de Churn

In [219]:
tabela["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [220]:
tabela["Churn"].value_counts(normalize=True) * 100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

In [221]:
tabela.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [222]:
tabela["FaixaTenure"] = pd.cut(
    tabela["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 meses", "13-24 meses", "25-48 meses", "49-72 meses"]
)

In [223]:
import plotly.express as px

In [224]:
fig = px.histogram(
    tabela,
    x="Churn",
    text_auto=True,
    title="Distribuição de Churn",
    labels={"Churn": "Cancelamento"}
)

fig.update_yaxes(title="Quantidade de Clientes")

fig.show()

## 4. Análise dos Fatores Associados ao Churn

In [225]:
churn_contrato = (
    tabela.groupby("Contract")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Taxa de Churn")
)

fig = px.bar(
    churn_contrato,
    x="Contract",
    y="Taxa de Churn",
    text_auto=".2f",
    title="Taxa de Churn por Tipo de Contrato",
    labels={
        "Contract": "Tipo de Contrato",
        "Taxa de Churn": "Taxa de Churn (%)"
    }
)

fig.show()

In [226]:
churn_tenure = (
    tabela.groupby("FaixaTenure", observed=True)["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Taxa de Churn")
)

fig = px.bar(
    churn_tenure,
    x="FaixaTenure",
    y="Taxa de Churn",
    text_auto=".2f",
    title="Taxa de Churn por Tempo de Permanência",
    labels={
        "FaixaTenure": "Tempo de Permanência",
        "Taxa de Churn": "Taxa de Churn (%)"
    }
)

fig.show()

In [227]:
churn_internet = (
    tabela.groupby("InternetService")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Taxa de Churn")
)

fig = px.bar(
    churn_internet,
    x="InternetService",
    y="Taxa de Churn",
    text_auto=".2f",
    title="Taxa de Churn por Tipo de Serviço de Internet",
    labels={
        "InternetService": "Serviço de Internet",
        "Taxa de Churn": "Taxa de Churn (%)"
    }
)

fig.show()

In [228]:
pd.crosstab(
    tabela[tabela["InternetService"] == "Fiber optic"]["Contract"],
    tabela[tabela["InternetService"] == "Fiber optic"]["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
Contract,,
Month-to-month,45.394737,54.605263
One year,80.705009,19.294991
Two year,92.773893,7.226107


In [229]:
pd.crosstab(
    tabela[tabela["InternetService"] == "Fiber optic"]["TechSupport"],
    tabela[tabela["InternetService"] == "Fiber optic"]["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
TechSupport,,
No,50.627803,49.372197
Yes,77.367206,22.632794


In [230]:
fibra = tabela[tabela["InternetService"] == "Fiber optic"]

pd.crosstab(
    [fibra["Contract"], fibra["TechSupport"]],
    fibra["Churn"],
    normalize="index"
) * 100

Churn                              No        Yes
Contract       TechSupport                      
Month-to-month No           42.483296  57.516704
               Yes          61.144578  38.855422
One year       No           81.469649  18.530351
               Yes          79.646018  20.353982
Two year       No           91.735537   8.264463
               Yes          93.181818   6.818182

In [231]:
pd.crosstab(
    [fibra["Contract"], fibra["TechSupport"]],
    fibra["Churn"]
)

Churn                        No   Yes
Contract       TechSupport           
Month-to-month No           763  1033
               Yes          203   129
One year       No           255    58
               Yes          180    46
Two year       No           111    10
               Yes          287    21

In [232]:
churn_pagamento = (
    tabela.groupby("PaymentMethod")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Taxa de Churn")
)

fig = px.bar(
    churn_pagamento,
    x="PaymentMethod",
    y="Taxa de Churn",
    text_auto=".2f",
    title="Taxa de Churn por Forma de Pagamento",
    labels={
        "PaymentMethod": "Forma de Pagamento",
        "Taxa de Churn": "Taxa de Churn (%)"
    }
)

fig.show()

In [233]:
pd.crosstab(
    [tabela["Contract"], tabela["PaymentMethod"]],
    tabela["Churn"],
    normalize="index"
) * 100

Churn                                            No        Yes
Contract       PaymentMethod                                  
Month-to-month Bank transfer (automatic)  65.874363  34.125637
               Credit card (automatic)    67.219153  32.780847
               Electronic check           46.270270  53.729730
               Mailed check               68.421053  31.578947
One year       Bank transfer (automatic)  90.281330   9.718670
               Credit card (automatic)    89.698492  10.301508
               Electronic check           81.556196  18.443804
               Mailed check               93.175074   6.824926
Two year       Bank transfer (automatic)  96.631206   3.368794
               Credit card (automatic)    97.762478   2.237522
               Electronic check           92.261905   7.738095
               Mailed check               99.214660   0.785340

In [234]:
pd.crosstab(
    tabela["OnlineSecurity"],
    tabela["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
OnlineSecurity,,
No,58.233276,41.766724
No internet service,92.595020,7.404980
Yes,85.388806,14.611194


## 5. Principais Achados e Recomendações

### Principais achados

- **Contrato mensal:** apresentou 42,71% de churn, contra 11,27% no contrato de 1 ano e 2,83% no contrato de 2 anos.
- **Baixo tempo de permanência:** clientes com até 12 meses apresentaram 47,44% de churn.
- **Fibra óptica:** apresentou 41,89% de churn, acima dos 18,96% observados no DSL.
- **Ausência de suporte técnico:** entre clientes de fibra óptica, aqueles sem suporte apresentaram 49,37% de churn.
- **Electronic check:** apresentou 45,29% de churn, a maior taxa entre as formas de pagamento analisadas.
- **OnlineSecurity:** clientes sem segurança online apresentaram 41,77% de churn, contra 14,61% entre aqueles que possuem o serviço.

Esses resultados indicam associações entre características dos clientes e o cancelamento, mas não estabelecem relação de causa e efeito.

## Recomendações de Negócio

- **Incentivar contratos de maior duração:** oferecer benefícios para clientes de contrato mensal migrarem para planos anuais, já que esse grupo apresentou maior churn.

- **Priorizar a retenção nos primeiros 12 meses:** criar ações de acompanhamento e fidelização para novos clientes, período em que foi observada a maior taxa de cancelamento.

- **Investigar a experiência dos clientes de fibra óptica:** esse grupo apresentou churn elevado, especialmente entre clientes com contrato mensal e sem suporte técnico.

- **Fortalecer o suporte técnico:** oferecer ou incentivar a contratação de suporte, principalmente para clientes de fibra óptica.

- **Avaliar clientes que utilizam Electronic check:** investigar por que essa forma de pagamento está associada a uma taxa de churn mais alta e incentivar métodos automáticos de pagamento quando apropriado.

- **Avaliar serviços adicionais de segurança:** clientes sem OnlineSecurity apresentaram maior churn, indicando uma oportunidade para investigar se pacotes com serviços adicionais estão relacionados a maior retenção.